# OULAD source audit

This notebook profiles the seven raw Open University Learning Analytics Dataset files before loading them into PostgreSQL. It checks structure, missingness, candidate keys, and the additive grain of VLE click activity. Raw CSV files are intentionally excluded from Git.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/raw/oulad")
EXPECTED_FILES = [
    "courses",
    "assessments",
    "studentInfo",
    "studentRegistration",
    "studentAssessment",
    "vle",
    "studentVle",
]

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Place the OULAD CSV files in {DATA_DIR.resolve()} before running this notebook."
    )

## 1. File inventory and schema profile

In [2]:
tables = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in EXPECTED_FILES}

inventory = pd.DataFrame(
    {
        "rows": {name: len(frame) for name, frame in tables.items()},
        "columns": {name: frame.shape[1] for name, frame in tables.items()},
        "duplicate_rows": {name: int(frame.duplicated().sum()) for name, frame in tables.items()},
        "missing_cells": {name: int(frame.isna().sum().sum()) for name, frame in tables.items()},
    }
).sort_index()

inventory

,rows,columns,duplicate_rows,missing_cells
assessments,206,6,0,11
courses,22,3,0,0
studentAssessment,173912,5,0,173
studentInfo,32593,12,0,1111
studentRegistration,32593,5,0,22566
studentVle,10655280,6,787170,0
vle,6364,6,0,10486


In [3]:
column_profile = pd.concat(
    {
        name: pd.DataFrame(
            {
                "dtype": frame.dtypes.astype(str),
                "missing": frame.isna().sum(),
                "unique": frame.nunique(dropna=True),
            }
        )
        for name, frame in tables.items()
    },
    names=["table", "column"],
)

column_profile

dtype  missing  unique
table               column                                              
courses             code_module                  object        0       7
                    code_presentation            object        0       4
                    module_presentation_length    int64        0       7
assessments         code_module                  object        0       7
                    code_presentation            object        0       4
                    id_assessment                 int64        0     206
                    assessment_type              object        0       3
                    date                        float64       11      74
                    weight                      float64        0      24
studentInfo         code_module                  object        0       7
                    code_presentation            object        0       4
                    id_student                    int64        0   28785
                    gender                       object        0       2
                    region                       object        0      13
                    highest_education            object        0       5
                    imd_band                     object     1111      10
                    age_band                     object        0       3
                    num_of_prev_attempts          int64        0       7
                    studied_credits               int64        0      61
                    disability                   object        0       2
                    final_result                 object        0       4
studentRegistration code_module                  object        0       7
                    code_presentation            object        0       4
                    id_student                    int64        0   28785
                    date_registration           float64       45     332
                    date_unregistration         float64    22521     416
studentAssessment   id_assessment                 int64        0     188
                    id_student                    int64        0   23369
                    date_submitted                int64        0     312
                    is_banked                     int64        0       2
                    score                       float64      173     101
vle                 id_site                       int64        0    6364
                    code_module                  object        0       7
                    code_presentation            object        0       4
                    activity_type                object        0      20
                    week_from                   float64     5243      30
                    week_to                     float64     5243      30
studentVle          code_module                  object        0       7
                    code_presentation            object        0       4
                    id_student                    int64        0   26074
                    id_site                       int64        0    6268
                    date                          int64        0     295
                    sum_click                     int64        0     498

## 2. Candidate-key checks

Enrollment is defined by student, module, and presentation. Assessment submissions are expected to be unique for each student-assessment pair.

In [4]:
enrollment_key = ["code_module", "code_presentation", "id_student"]
assessment_submission_key = ["id_assessment", "id_student"]
course_presentation_key = ["code_module", "code_presentation"]

key_checks = pd.Series(
    {
        "studentInfo enrollment duplicates": tables["studentInfo"].duplicated(enrollment_key).sum(),
        "studentRegistration enrollment duplicates": tables["studentRegistration"].duplicated(enrollment_key).sum(),
        "studentAssessment submission duplicates": tables["studentAssessment"].duplicated(assessment_submission_key).sum(),
        "courses presentation duplicates": tables["courses"].duplicated(course_presentation_key).sum(),
    },
    name="duplicate_keys",
)

key_checks

studentInfo enrollment duplicates            0
studentRegistration enrollment duplicates    0
studentAssessment submission duplicates      0
courses presentation duplicates              0
Name: duplicate_keys, dtype: int64

## 3. VLE activity grain

The raw clickstream can contain multiple additive observations for the same student, resource, and relative day. These rows must be summed for the analytical fact table; they must not be removed as ordinary duplicates.

In [5]:
vle_activity_key = [
    "code_module",
    "code_presentation",
    "id_student",
    "id_site",
    "date",
]

vle_key_profile = (
    tables["studentVle"]
    .groupby(vle_activity_key, as_index=False)
    .agg(source_rows=("sum_click", "size"), total_clicks=("sum_click", "sum"))
)

vle_key_profile["source_rows"].value_counts().sort_index().head(10)

source_rows
1     6844815
2     1311841
3      168742
4       41322
5       40822
6       51410
7         275
8          68
9          20
10          5
Name: count, dtype: int64

## 4. Audit findings used in the model

- The source contains 32,593 enrollment rows and 28,785 unique students.
- The seven modules appear in 22 course presentations.
- `studentAssessment` contains 173,912 submissions; 173 score values are missing.
- `studentVle` contains 10,655,280 rows and no missing values.
- `studentInfo.imd_band` contains 1,111 missing values.
- Missing exam due dates and missing unregistration dates are valid source semantics and remain nullable.
- The audited enrollment, course-presentation, and student-assessment candidate keys contain no duplicates.